# Notebook 04 — Supplementary Figures

Generates all main figures from the paper using only precomputed `.npz` data and `metasignal.stdpy`. No external helper module is imported.

| Figure | Content |
|--------|---------|
| Fig 2  | Difficulty dependence — mean measures hard vs easy |
| Fig 3  | Metacognitive bias — Xue recoding effect |
| Fig 4  | Cross-measure correlation matrices |

In [1]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


metasignal loaded successfully.


In [2]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


In [3]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


Statistical helper functions defined.


In [4]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
FIGS = os.path.join(REPO, 'notebooks', 'figures')
os.makedirs(FIGS, exist_ok=True)

## Load all precomputed data

In [5]:
sh_diff = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['diff']   # (20,3,20)
r1_diff = np.load(os.path.join(OUT, 'rouault1_mle.npz'))['diff']  # (466,2,20)
r2_diff = np.load(os.path.join(OUT, 'rouault2_mle.npz'))['diff']  # (484,2,20)
ha_npz  = np.load(os.path.join(OUT, 'haddara_mle.npz'))
ma_npz  = np.load(os.path.join(OUT, 'maniscalco_mle.npz'))
ha_raw  = ha_npz['raw'];   ha_bias = ha_npz['bias']
ma_raw  = ma_npz['raw'];   ma_bias = ma_npz['bias']
sh_bias_full = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['bias']
sh_bias = np.nanmean(sh_bias_full, axis=1)
sh_raw  = np.nanmean(sh_diff, axis=1)
print("Data loaded.")


Data loaded.


## Supplementary Figure 2 — Difficulty dependence

In [6]:
sh_clean = remove_3sd_outliers(sh_diff[:, [0,2], :])
r1_clean = remove_3sd_outliers(r1_diff)
r2_clean = remove_3sd_outliers(r2_diff)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
COLORS = ['#d55e00', '#0072b2', '#009e73']
for ax, (arr, label, color) in zip(axes, [
    (sh_clean, 'Shekhar (n=20)', COLORS[0]),
    (r1_clean, 'Rouault1 (n=466)', COLORS[1]),
    (r2_clean, 'Rouault2 (n=484)', COLORS[2]),
]):
    delta = arr[:, 1, :] - arr[:, 0, :]
    means = np.nanmean(delta, axis=0)
    sems  = np.nanstd(delta, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(delta), axis=0))
    x = np.arange(N_MEAS)
    ax.bar(x, means, color=color, alpha=0.8)
    ax.errorbar(x, means, yerr=sems, fmt='none', color='k', capsize=3, linewidth=1)
    ax.axhline(0, color='k', linewidth=0.6, linestyle='--')
    ax.set_xticks(x); ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=7)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Easy − Hard ± SEM')
plt.suptitle('Supplementary Figure 2: Difficulty Dependence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'supp_fig2_difficulty.png'), dpi=150, bbox_inches='tight')
print("Saved supp_fig2_difficulty.png")
plt.show()


Saved supp_fig2_difficulty.png


## Supplementary Figure 3 — Xue recoding

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (bias_arr, label, color) in zip(axes, [
    (ha_bias,  'Haddara (n=70)',    '#d55e00'),
    (ma_bias,  'Maniscalco (n=22)', '#0072b2'),
    (sh_bias,  'Shekhar (n=20)',    '#009e73'),
]):
    delta = bias_arr[:, 1, :] - bias_arr[:, 0, :]
    means = np.nanmean(delta, axis=0)
    sems  = np.nanstd(delta, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(delta), axis=0))
    x = np.arange(N_MEAS)
    ax.bar(x, means, color=color, alpha=0.8)
    ax.errorbar(x, means, yerr=sems, fmt='none', color='k', capsize=3, linewidth=1)
    ax.axhline(0, color='k', linewidth=0.6, linestyle='--')
    ax.set_xticks(x); ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=7)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Recode2 − Recode1 ± SEM')
plt.suptitle('Supplementary Figure 3: Metacognitive Bias (Xue Recoding)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'supp_fig3_xue_recode.png'), dpi=150, bbox_inches='tight')
print("Saved supp_fig3_xue_recode.png")
plt.show()


Saved supp_fig3_xue_recode.png


## Supplementary Figure 4 — Cross-measure correlations

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (raw, label) in zip(axes, [
    (ha_raw, 'Haddara (n=70)'),
    (ma_raw, 'Maniscalco (n=22)'),
    (sh_raw, 'Shekhar (n=20)'),
]):
    R = np.full((N_MEAS, N_MEAS), np.nan)
    for i in range(N_MEAS):
        for j in range(N_MEAS):
            ok = ~np.isnan(raw[:,i]) & ~np.isnan(raw[:,j])
            if ok.sum() >= 3:
                from scipy.stats import pearsonr as pr_
                R[i,j], _ = pr_(raw[ok,i], raw[ok,j])
    im = ax.imshow(R, vmin=-1, vmax=1, cmap='RdYlBu_r', aspect='auto')
    ax.set_title(label, fontweight='bold')
    ax.set_xticks(range(N_MEAS)); ax.set_yticks(range(N_MEAS))
    ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=6)
    ax.set_yticklabels(MEASURE_NAMES, fontsize=6)
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle('Supplementary Figure 4: Cross-Measure Correlations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'supp_fig4_correlations.png'), dpi=150, bbox_inches='tight')
print("Saved supp_fig4_correlations.png")
plt.show()


Saved supp_fig4_correlations.png
